In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

path = "../data/raw/Mobile_Parts_Wholesale_Dataset.xlsx"
sales = pd.read_excel(path, sheet_name="Sales_Transactions")
sales['order_date'] = pd.to_datetime(sales['order_date'])

snapshot_date = sales['order_date'].max() + pd.Timedelta(days=1)

rfm = sales.groupby('customer_id').agg(
    recency=('order_date', lambda x: (snapshot_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('total_amount_inr', 'sum')
).reset_index()

print(rfm.shape)
rfm.head()

(180, 4)


,customer_id,recency,frequency,monetary
0,C0001,6,41,228329.0
1,C0002,5,41,200205.5
2,C0003,3,25,139561.0
3,C0004,8,74,362173.5
4,C0005,23,27,82187.5


In [2]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['segment'] = kmeans.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby('segment')[['recency','frequency','monetary']].mean()
print(cluster_summary)

           recency  frequency       monetary
segment                                     
0        14.986301  27.808219  134316.527397
1        69.000000  28.923077  122738.076923
2        13.093750  71.281250  347358.093750
3        13.048387  52.274194  239228.209677


D:\dataScience\envs\dss_project\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [3]:
segment_labels = {0: "High Value", 1: "Steady/Regular", 2: "New/Occasional", 3: "At Risk"}
rfm['segment_label'] = rfm['segment'].map(segment_labels)

rfm.to_csv("../reports/customer_segments.csv", index=False)
print("Saved customer_segments.csv")
rfm['segment_label'].value_counts()

Saved customer_segments.csv


segment_label
High Value        73
At Risk           62
New/Occasional    32
Steady/Regular    13
Name: count, dtype: int64